# Wie findet ein Fußballroboter den Ball? – Musterlösung

Ein Roboter bekommt ein Kamerabild. Du erkennst den Ball auf einen Blick –
aber wie könntest du einem Computer erklären, woran er ihn erkennen soll?
Wir entwickeln dafür Regeln für Farbe und Form und probieren sie an Bildern aus.

Du arbeitest in kleinen Schritten: **Bild verstehen → Farben auswählen → Flächen finden
→ Rechtecke zeichnen → Ergebnisse verbessern und prüfen.**

Dieses Notebook zeigt zu jedem Arbeitsschritt eine mögliche Lösung. Vergleiche sie mit
deiner eigenen Idee und beobachte die Zwischenbilder. Die Regeln sind keine allgemeingültige
Ballerkennung: Sie wurden für den orangefarbenen Ball entwickelt.

## Start: Notebook vorbereiten

Eine Zelle ist ein Abschnitt mit Text oder ausführbarem Python-Code. Führe Codezellen mit
**Shift + Enter** von oben nach unten aus. Nach einer Änderung musst du die Zelle und alle
davon abhängigen Zellen erneut ausführen. Eine Ausgabe unter einer Zelle zeigt das Ergebnis.

Die folgenden eingeklappten Zellen laden Werkzeuge und Bilder. **Nur ausführen, nichts bearbeiten.**
Falls deine Oberfläche den Code trotzdem zeigt, kannst du ihn für die Aufgabe übergehen.
Hinweise zur Einrichtung stehen in der README; bei einer Fehlermeldung hilft die Betreuung.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

for candidate in (Path.cwd(), *Path.cwd().parents):
    for helper_dir in (candidate, candidate / "tutorials" / "schnupperstudium"):
        if (helper_dir / "vision_workshop.py").is_file():
            sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise FileNotFoundError("vision_workshop.py fehlt. Bitte vom Repository oder Notebook-Ordner aus starten.")

In [ ]:
from download_datasets import ensure_datasets
from vision_workshop import evaluate, load_dataset, plot_metrics, print_evaluation
from workshop_plots import show_boxes, show_channels, show_hsv_cake, show_image, show_mask

ensure_datasets(
    {
        "orange": "https://data.bit-bots.de/Schnupperstudium2026/orange.tar.gz",
        "realistic": "https://data.bit-bots.de/Schnupperstudium2026/realistic.tar.gz",
    }
)
train_records = load_dataset("orange", split="train", limit=32)
plt.rcParams["figure.dpi"] = 100
print("Bereit! Wähle im nächsten Schritt ein Bild aus.")

## Ein Bild untersuchen

Die vorbereitete Liste `train_records` enthält Übungsbilder. Mit einem Index wählst du
einen Eintrag aus; Python beginnt beim Zählen bei null. `record` ist dieser Eintrag,
`image_bgr` sein Bild. Eine Variable ist ein Name, unter dem wir einen Wert aufbewahren.
`show_image(...)` zeigt das übergebene Bild an; diese Zeichenhilfe ist bereits vorbereitet.

**Aufgabe:** Betrachte verschiedene Bilder. Welche Merkmale unterscheiden den Ball vom
Hintergrund? Findest du auch Bilder ohne Ball? Notiere eine erste Regel in eigenen Worten.

In [ ]:
bild_index = 0
record = train_records[bild_index]
image_bgr = record["image_bgr"]
show_image(image_bgr)
plt.show()

### Pixel, x und y

Ein digitales Bild besteht aus kleinen Bildpunkten, den **Pixeln**. Jeder Pixel hat eine
Position und eine Farbe. Der Ursprung liegt **oben links**: `x` wächst nach rechts,
`y` nach unten. Das unterscheidet sich von vielen Koordinatensystemen aus dem Matheunterricht.

Python speichert das Bild zeilenweise. Deshalb steht beim Zugriff `image_bgr[y, x]`
zuerst die Zeile und dann die Spalte. `.shape` beschreibt die Größe des Bildarrays:
Höhe, Breite und Anzahl der Farbkanäle. Ein Array ist eine geordnete Sammlung von Zahlen.

**Aufgabe:** Lies die Bildgröße ab. Wähle danach mit den Achsen im Bild einen Punkt auf
dem Ball und einen im Hintergrund. Was verändert sich an der ausgegebenen Farbe?

In [ ]:
hoehe, breite, kanaele = image_bgr.shape
print("Höhe:", hoehe, "Breite:", breite, "Farbkanäle:", kanaele)
x = breite // 2
y = hoehe // 2
print("Farbe am gewählten Punkt (BGR):", image_bgr[y, x])

### Den Ball mit einem Rechteck beschreiben

Unser Programm soll die Position des Balls liefern. Dafür verwenden wir ein möglichst enges,
achsenparalleles Rechteck um den Ball: eine **Bounding Box**, kurz **Box**.
Wir speichern sie als `[links, oben, rechts, unten]`, also die x- und y-Koordinaten
der linken oberen und rechten unteren Ecke. Die rechte und untere Grenze liegen direkt
hinter den eingeschlossenen Pixeln. Breite ist `rechts - links`, Höhe ist `unten - oben`.

`record["boxes"]` enthält die von Menschen vorgegebenen Vergleichsrechtecke, die
**Referenzboxen**. Sie helfen später beim Prüfen; unser Detektor darf sie nicht nachschlagen.
Mehrere Bälle brauchen mehrere Boxen. Ohne Ball ist die Liste leer.

**Aufgabe:** Ersetze die Eckpunkte unten durch ein enges Rechteck um einen sichtbaren Ball.
Vergleiche dein magentafarbenes Rechteck mit dem grünen Referenzrechteck.

In [ ]:
meine_boxen = [[0, 0, breite, hoehe]]
show_boxes(record, meine_boxen)
plt.show()

## Farben als Zahlen verstehen

**RGB** beschreibt eine Farbe durch ihre Anteile an Rot, Grün und Blau.
Große Werte bedeuten einen starken Anteil, kleine einen schwachen. Gleich große Anteile
ergeben einen Grauton. In unseren Bildern werden die Anteile als ganze Zahlen gespeichert.

Das Bildverarbeitungswerkzeug **OpenCV** liefert dieselben Anteile in der Reihenfolge
**BGR**: Blau, Grün, Rot. Darum heißt unsere Bildvariable `image_bgr`.
Die vorbereiteten Zeichenhilfen drehen die Reihenfolge für die Anzeige passend um.

Die folgenden Graubilder zeigen jeden Farbanteil einzeln. Helle Stellen haben einen
großen Anteil des jeweiligen Kanals.
**Aufgabe:** In welchen Kanälen hebt sich der orange Ball vom grünen Feld ab?

In [ ]:
print("Wertebereich je Farbkanal:", np.iinfo(image_bgr.dtype).min, "bis", np.iinfo(image_bgr.dtype).max)
show_channels(image_bgr)
plt.show()

### HSV: Farbe, Sättigung und Helligkeit trennen

Für „orange, auch wenn es etwas dunkler ist“ ist eine andere Darstellung hilfreich:

- **H – Hue (Farbton):** Welche Farbe? Die Farbtöne liegen rund um die Torte.
- **S – Saturation (Sättigung):** Wie kräftig ist die Farbe? In der Mitte ist sie grau,
  zum Rand hin wird sie kräftiger.
- **V – Value (Helligkeit):** Wie hell ist sie? Die Scheiben zeigen verschiedene Helligkeiten.

**Aufgabe:** Suche Orange auf der Torte. Was passiert, wenn du bei gleichem Farbton zur
Mitte gehst? Was ändert sich auf einer dunkleren Scheibe?
Die Beschriftung zeigt die Zahlenbereiche der hier verwendeten OpenCV-Darstellung.
Der Farbton läuft im Kreis; am Übergang liegen auf beiden Seiten rote Farbtöne.

In [ ]:
show_hsv_cake()
plt.show()

`cv2.cvtColor` rechnet ein Bild in eine andere Farbdarstellung um.
`cv2.COLOR_BGR2HSV` sagt der Funktion, welche Umrechnung wir wollen.
Die Positionen der Pixel bleiben dabei gleich.

**Aufgabe:** Untersuche wieder Ball- und Hintergrundpixel, diesmal in HSV.
Sammle auch Werte von hellen und dunklen Stellen des Balls. Welche Bereiche könnten passen?

Die HSV-Zahlen sind keine RGB-Anteile und die H-Werte hier keine Gradangaben.
Für unsere Ganzzahlbilder zeigt die nächste Zelle die gültigen Bereiche.


In [ ]:
image_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
print("BGR am gewählten Punkt:", image_bgr[y, x])
print("HSV am gewählten Punkt:", image_hsv[y, x])
print("OpenCV-HSV: H von 0 bis 179; S und V von 0 bis 255")

## Passende Pixel auswählen: deine Farbmaske

Eine **Maske** ist ein Bild, das nur zwischen ausgewählten und nicht ausgewählten Pixeln
unterscheidet. Wir zeigen ausgewählte Pixel hell und die anderen dunkel.

`cv2.inRange(bild, untergrenze, obergrenze)` wählt Pixel aus, deren Kanalwerte jeweils
innerhalb der Grenzen liegen. Die Grenzen sind eingeschlossen.
`np.array([h, s, v])` packt die Grenzwerte in ein Array für OpenCV.
Die folgende Zelle legt zunächst die Grenzen fest. Verändere sie anhand deiner Beobachtungen.

In [ ]:
h_min, h_max = 0, 20
s_min, v_min = 140, 80
hsv_unten = np.array([h_min, s_min, v_min])
hsv_oben = np.array([h_max, 255, 255])

Mit `def` definieren wir eine **Funktion**: einen wiederverwendbaren Arbeitsschritt.
`make_mask` erhält ein Bild, wandelt es nach HSV um und gibt mit `return` eine Maske zurück.
Eingerückte Zeilen gehören zur Funktion. Erst ein Aufruf führt diese Zeilen aus.

**Aufgabe:** Die Lösung wählt mit `cv2.inRange` die passenden Pixel aus. Probiere andere Grenzen.

Führe anschließend die Anzeige aus. Ist der Ball hell? Welche anderen Flächen werden
ebenfalls ausgewählt? Prüfe deine Grenzen auf mehreren Übungsbildern.

In [ ]:
def make_mask(image_bgr):
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, hsv_unten, hsv_oben)
    return mask

In [ ]:
mask = make_mask(image_bgr)
show_mask(image_bgr, mask)
plt.show()

## Aus Flächen werden Boxen

Benachbarte helle Pixel bilden Flächen. Eine **Kontur** beschreibt den Rand einer solchen Fläche.
`cv2.findContours` sucht diese Ränder in der Maske. Mit `cv2.RETR_EXTERNAL` betrachten wir
nur die äußeren Ränder; `cv2.CHAIN_APPROX_SIMPLE` spart überflüssige Randpunkte.
Die Funktion liefert Konturen und Zusatzinformationen; `_` bedeutet hier, dass wir die
Zusatzinformationen nicht brauchen.

`cv2.boundingRect(kontur)` liefert `links, oben, breite, hoehe` eines umschließenden Rechtecks.
Unser Boxformat benötigt stattdessen die Endkoordinaten. Wie berechnest du rechts und unten?

In der Funktion beginnt `boxes = []` mit einer leeren Liste. Die `for`-Schleife bearbeitet
jede Kontur nacheinander; `boxes.append(...)` fügt ein Rechteck hinzu.
Am Ende wandeln wir die Liste in ein Array um: jede Zeile eine Box, jede Spalte eine
Eckkoordinate. `reshape(-1, 4)` sichert dieses Format auch für eine leere Liste.

**Aufgabe:** Verfolge den Weg von der Kontur zur Box in der Lösung.
Prüfe die Ausgabe: Welche Flächen sind wirklich Bälle? Welche sind Fehlmeldungen?

In [ ]:
def boxes_from_mask(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for contour in contours:
        left, top, width, height = cv2.boundingRect(contour)
        boxes.append([left, top, left + width, top + height])
    return np.asarray(boxes, dtype=float).reshape(-1, 4)

In [ ]:
predicted = boxes_from_mask(make_mask(image_bgr))
show_boxes(record, predicted)
plt.show()

## Die Schritte zu einem Detektor verbinden

`detect_balls` erhält nur das Kamerabild. Sie ruft deine Maskenfunktion und danach deine
Boxenfunktion auf. Die Referenzboxen werden ausschließlich zum Vergleichen verwendet.

**Aufgabe:** Probiere den Detektor auf verschiedenen Bildern aus, auch auf Bildern ohne Ball.
Führe beim Wechsel des Bildes die Zelle zur Bildauswahl und diese Anzeige erneut aus.
Notiere einen gelungenen Treffer und eine Fehlentscheidung.

In [ ]:
def detect_balls(image_bgr):
    return boxes_from_mask(make_mask(image_bgr))


show_boxes(record, detect_balls(image_bgr))
plt.show()

## Ergebnisse prüfen und verbessern

Wir unterscheiden **Treffer**, **Fehlmeldungen** und **übersehene Bälle**.
Für einen Treffer muss ein vorhergesagtes Rechteck ausreichend mit einer Referenzbox
überlappen. Dazu teilt die Auswertung die gemeinsame Fläche durch die insgesamt von beiden
Rechtecken bedeckte Fläche: **Intersection over Union (IoU)**.
Eine perfekte Überdeckung liefert den größten möglichen Wert, getrennte Boxen den kleinsten.
Jeder Ball und jede Vorhersage können nur zu einem Treffer gehören.

Die Kennzahlen beantworten unterschiedliche Fragen:

- **Precision:** Wie viele der gemeldeten Bälle sind richtig?
- **Recall:** Wie viele der vorhandenen Bälle haben wir gefunden?
- **F1:** Ein gemeinsamer Wert, der nur gut ist, wenn Precision und Recall gut sind.

`TP`, `FP` und `FN` in der Ausgabe stehen für Treffer, Fehlmeldungen und übersehene Bälle.
Betrachte auch `real` (Kamera) und `simulation` getrennt.
**Aufgabe:** Welche Fehler treten auf? Ändere jeweils nur eine Idee und vergleiche erneut.
Die IoU-Schwelle ist Teil der festen Auswertung; verändere sie nicht, um das Ergebnis zu verbessern.

In [ ]:
IOU_THRESHOLD = 0.5
train_result = evaluate(train_records, detect_balls, threshold=IOU_THRESHOLD)
print_evaluation(train_result, "Übungsbilder")
plot_metrics(train_result)
plt.show()

Die nächste Anzeige zeigt Beispiele, bei denen tatsächlich ein Fehler aufgetreten ist.
Die Maske wird von **deiner aktuellen Maskenfunktion** berechnet. Führe nach einer Änderung
zuerst die Auswertung oben erneut aus, damit Masken und Boxen zum gleichen Stand gehören.

In [ ]:
failures = [item for item in train_result["items"] if item["fp"] or item["fn"]]
for item in failures[:3]:
    show_boxes(item["record"], item["predicted"])
    show_mask(item["record"]["image_bgr"], make_mask(item["record"]["image_bgr"]))
    plt.show()
if not failures:
    print("In dieser Auswahl sind keine Fehler aufgetreten.")

## Vertiefung: Störende Flecken entfernen

Diesen Abschnitt kannst du überspringen. Die Grundidee funktioniert bereits ohne ihn.

Die Farbmaske enthält vielleicht kleine helle Flecken oder Lücken in der Ballfläche.
**Öffnen** entfernt kleine Flecken; **Schließen** kann kleine Lücken füllen.
Dabei verschiebt OpenCV eine kleine Nachbarschaft, den **Kernel**, über die Maske.
Ein größerer Kernel verändert größere Strukturen und kann auch kleine Bälle entfernen.

**Experiment:** Die folgende Zelle zeigt nur einen Vergleich. Ändere die Kernelgröße und
probiere `cv2.MORPH_CLOSE` statt `cv2.MORPH_OPEN` aus. Was verbessert sich, was geht verloren?
Wenn es hilft, übernimm die Operation vor `return mask` in deine Funktion `make_mask`.
Ohne diese Übernahme bleibt dein Detektor unverändert.

In [ ]:
kernel_size = 3
kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
original_mask = make_mask(image_bgr)
cleaned_mask = cv2.morphologyEx(original_mask, cv2.MORPH_OPEN, kernel)
show_mask(image_bgr, original_mask)
show_mask(image_bgr, cleaned_mask)
plt.show()

## Vertiefung: Welche Formen passen zu einem Ball?

Eine orange Fläche muss kein Ball sein. Wir können zusätzlich prüfen:

- **Fläche:** Ist der Fleck groß genug? Zu strenge Grenzen entfernen entfernte Bälle.
- **Seitenverhältnis:** Ist das Rechteck eher kompakt oder lang und schmal?
- **Rundheit:** Wie groß ist die Fläche im Vergleich zum Umfang?

Für die Rundheit verwenden wir $4\pi A/U^2$, mit Fläche $A$ und Umfang $U$.
Setzt man die Kreisformeln ein, erhält man den Idealwert eins. Langgezogene oder zerklüftete
Flächen haben kleinere Werte. Auch ein Ball kann wegen Verdeckung oder unsauberer
Farbauswahl eine unregelmäßige Kontur haben.

`cv2.contourArea` misst die Konturfläche, `cv2.arcLength(contour, True)` den geschlossenen
Umfang. `continue` überspringt eine ungeeignete Kontur und geht zur nächsten weiter.
Ein Umfang von null muss vor der Division ausgeschlossen werden.

**Aufgabe:** Diese Variante ergänzt die Grundlösung. Vergleiche ihre Wirkung und ändere jeweils nur eine Grenze.

Für das Seitenverhältnis kannst du `min(width / height, height / width)` verwenden.
Es wird bei schmalen Rechtecken klein und behandelt breite und hohe Rechtecke gleich.

In [ ]:
min_area = 60
min_roundness = 0.5
min_aspect = 0.4


def boxes_from_mask(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for contour in contours:
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        if area < min_area or perimeter == 0:
            continue
        roundness = 4 * np.pi * area / perimeter**2
        if roundness < min_roundness:
            continue
        left, top, width, height = cv2.boundingRect(contour)
        aspect = min(width / height, height / width)
        if aspect < min_aspect:
            continue
        boxes.append([left, top, left + width, top + height])
    return np.asarray(boxes, dtype=float).reshape(-1, 4)

Diese Referenz verwendet die auf Trainingsbildern ausgewählten Grenzen und keine
Maskenbereinigung. `detect_balls` ruft automatisch die zuletzt ausgeführte Definition von
`boxes_from_mask` auf. Der Tuning-Bericht und das zugehörige Skript sind für Betreuende
in der README verlinkt.

## Erst entwickeln, dann an unbekannten Bildern testen

Bisher haben wir Bilder verwendet, auf denen wir unsere Regeln verbessern dürfen:
die **Trainingsbilder** (`train`). Auch ohne neuronales Netz passen wir unsere Lösung an
diese Beispiele an. Die kleine Auswahl hilft beim Ausprobieren; jetzt prüfen wir alle
Trainingsbilder. Das Ergebnis kann dadurch anders ausfallen.

**Testbilder** (`test`) sind für die abschließende Prüfung reserviert. Sie zeigen,
ob die Regeln auch auf bisher nicht verwendeten Bildern funktionieren.
Lege deshalb zuerst deine endgültige Lösung fest. Wenn du nach dem Test weiter verbesserst,
ist derselbe Test kein unabhängiger Abschlussvergleich mehr.

In [ ]:
all_train_records = load_dataset("orange", split="train", limit=None)
full_train_result = evaluate(all_train_records, detect_balls, threshold=IOU_THRESHOLD)
print_evaluation(full_train_result, "Alle Trainingsbilder")
plot_metrics(full_train_result)
plt.show()

**Abschlusstest:** Wenn deine Lösung feststeht, setze `RUN_FINAL_TEST` auf `True`.
Vergleiche besonders echte und simulierte Bilder. Ein gutes Ergebnis auf diesen Bildern
bedeutet noch nicht, dass alle Kameras, Beleuchtungen und Spielsituationen abgedeckt sind.

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    test_records = load_dataset("orange", split="test", limit=None)
    test_result = evaluate(test_records, detect_balls, threshold=IOU_THRESHOLD)
    print_evaluation(test_result, "Abschlusstest")
    plot_metrics(test_result)
    plt.show()
else:
    print("Lege zuerst deine Lösung fest. Danach kannst du den Test einschalten.")

## Funktioniert die Idee auch bei anderen Bällen?

Im Datensatz `realistic` gibt es auch weiße oder gemusterte Bälle.
**Vermutung:** Welche unserer Annahmen passt dort nicht mehr?

Schalte den Transfer mit `RUN_TRANSFER` ein und verwende deine fertige Funktion unverändert.
Ein schlechtes Ergebnis kann hier aufschlussreich sein: Welche andere Information als
Orange könnte helfen? Für die Entwicklung einer neuen Lösung wären wieder eigene
Trainingsbilder nötig.

In [ ]:
RUN_TRANSFER = False

if RUN_TRANSFER:
    transfer_records = load_dataset("realistic", split="test", limit=None)
    transfer_result = evaluate(transfer_records, detect_balls, threshold=IOU_THRESHOLD)
    print_evaluation(transfer_result, "Andere Bälle")
    for item in transfer_result["items"][:4]:
        show_boxes(item["record"], item["predicted"])
        plt.show()
else:
    print("Probiere den Transfer nach dem Abschlusstest aus.")

## Was hast du herausgefunden?

Ergänze deine Antworten hier:

- Meine erste Idee zur Ballerkennung war …
- Eine hilfreiche Änderung war …, weil …
- Ein Beispiel, bei dem mein Programm scheitert, ist …
- Echte Bilder und Simulationsbilder unterscheiden sich darin, dass …
- Bei anderen Bällen passt folgende Annahme nicht mehr: …
- Als Nächstes würde ich ausprobieren, …